In [3]:
import os

import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark import SparkConf
from pyspark.sql import Row, SparkSession

In [4]:
conf = (
    SparkConf()
    .setAppName("[ICAI] DataFrame: A fondo")
    .set("spark.jars", "/var/lib/sqoop/mysql-connector-java-5.1.44-bin.jar")
    .set("spark.executor.memory", "4g")
    .set("spark.executor.cores", "2")

)

In [5]:
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [6]:
ventas = spark.read.load("/datos/examen_24/ventas.parquet").cache()
ventas.show(10)

+--------+----------+-----------+----------+
|venta_id|cliente_id|producto_id|     fecha|
+--------+----------+-----------+----------+
|  179201|       181|       1053|2024-07-28|
|  179202|       173|       1054|2024-07-28|
|  179203|       169|       1044|2024-07-28|
|  179204|       168|       1011|2024-07-28|
|  179205|       190|       1058|2024-07-28|
|  179206|       115|       1057|2024-07-28|
|  179207|       157|       1024|2024-07-28|
|  179208|       120|       1036|2024-07-28|
|  179209|       190|       1051|2024-07-28|
|  179210|       143|       1037|2024-07-28|
+--------+----------+-----------+----------+
only showing top 10 rows



In [7]:
ventas.printSchema()

root
 |-- venta_id: long (nullable = true)
 |-- cliente_id: long (nullable = true)
 |-- producto_id: long (nullable = true)
 |-- fecha: date (nullable = true)



In [8]:
# Fecha de la primera compra
ventas.agg(F.min("fecha")).show(1)

+----------+
|min(fecha)|
+----------+
|2024-01-01|
+----------+



In [9]:
#Fecha de la útima compra
ventas.agg(F.max("fecha")).show(1)

+----------+
|max(fecha)|
+----------+
|2024-12-15|
+----------+



In [10]:
#Mes con más compras y cuántas son
ventas.withColumn("mes", F.month("fecha")).groupBy("mes").agg(
    F.count("*").alias("Conteo")).orderBy(F.desc("Conteo")).limit(1).show()

+---+------+
|mes|Conteo|
+---+------+
|  8| 26850|
+---+------+



In [11]:
productos = spark.read.options(header=True, inferSchema=True, sep="|").csv("/datos/examen_24/productos.tab")
productos.show(10)

+-----------+--------------------+-------+-----------------+
|producto_id|     nombre_producto| precio|          seccion|
+-----------+--------------------+-------+-----------------+
|       1001|           iPhone 13|2682.78|      Electrónica|
|       1007|         Dell XPS 13|1619.46|      Electrónica|
|       1019|     HP Spectre x360|1707.05|Electrodomésticos|
|       1025|       Beats Studio3| 2474.1|Electrodomésticos|
|       1034|           OnePlus 9|2560.46|Electrodomésticos|
|       1046|       Dyson Airwrap|2769.33|Electrodomésticos|
|       1040|         Philips Hue| 650.45|Electrodomésticos|
|       1060|     Columbia Jacket| 3211.1|       Moda Mujer|
|       1002|  Samsung Galaxy S21|1891.78|      Electrónica|
|       1020|Microsoft Surface...|3628.57|Electrodomésticos|
+-----------+--------------------+-------+-----------------+
only showing top 10 rows



In [12]:
productos.printSchema()

root
 |-- producto_id: integer (nullable = true)
 |-- nombre_producto: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- seccion: string (nullable = true)



In [13]:
#Hacemos un join
tabla_join = ventas.join(productos, "producto_id")
tabla_join.show(10)

+-----------+--------+----------+----------+--------------------+-------+-----------------+
|producto_id|venta_id|cliente_id|     fecha|     nombre_producto| precio|          seccion|
+-----------+--------+----------+----------+--------------------+-------+-----------------+
|       1053|  179201|       181|2024-07-28|Wilson Tennis Racket|1130.96|       Moda Mujer|
|       1054|  179202|       173|2024-07-28|     Fitbit Charge 4|3472.49|       Moda Mujer|
|       1044|  179203|       169|2024-07-28|   Bissell CrossWave|2817.23|Electrodomésticos|
|       1011|  179204|       168|2024-07-28|   Kindle Paperwhite|3647.34|      Electrónica|
|       1058|  179205|       190|2024-07-28|   Reebok Sports Bra|1161.28|       Moda Mujer|
|       1057|  179206|       115|2024-07-28|  Puma Running Shoes|3847.17|       Moda Mujer|
|       1024|  179207|       157|2024-07-28|          JBL Flip 5|2753.04|Electrodomésticos|
|       1036|  179208|       120|2024-07-28|      Oculus Quest 2|3360.55|Electro

In [16]:
###Cliente que más ha gastado en moda mujer.
tabla_join.filter(F.col("seccion")=="Moda Mujer").groupBy("cliente_id").agg(
    F.sum("precio").alias("Gasto Total")
).orderBy(F.desc("Gasto Total")).limit(1).show()

+----------+-----------------+
|cliente_id|      Gasto Total|
+----------+-----------------+
|       197|1210444.709999999|
+----------+-----------------+

